# 🧠 Training GRU BISINDO — Colab (replika persis repo)

Notebook ini menjalankan **logika training yang identik** dengan `src/gru_manager.py`
di repo (arsitektur, hyperparameter, pipeline data, augmentasi, format checkpoint).
Modul source di-*embed* apa adanya (byte-for-byte) ke `/content/src`.

## Cara pakai
1. **Runtime → Change runtime type → GPU** (T4 cukup).
2. Taruh dataset di Google Drive dengan struktur **sama persis** seperti di repo:
   `MyDrive/dataset_parquets/<schema>/<label>.parquet`
   (schema: `smart180`, `khukuh1629`, `adi1662`, `smart180_face1584`).
3. Jalankan cell berurutan: cek env → mount Drive → tulis source → konfigurasi →
   copy dataset → training → simpan ke Drive.

## Catatan
- Setiap epoch **selalu ada cek validasi** (val split) untuk early-stopping & pemilihan
  checkpoint terbaik (`best_val_acc`) — sama persis dengan repo.
- Varian `adi` (custom ReLU-GRU, loop per-timestep, 60 frame) lebih lambat daripada
  `khukuh`/`hybrid` (cuDNN `nn.GRU`, 30 frame).
- Toggle augmentasi hanya beneran ngefek di `smart180_face1584` (satu-satunya schema
  yang punya sample augmented di parquet).


In [ ]:
#@title 1. Cek environment & GPU
import sys, platform
print("Python:", sys.version.split()[0], "|", platform.platform())
try:
    import torch
    print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("⚠️  GPU OFF — Runtime > Change runtime type > GPU, lalu Restart")
except Exception as e:
    print("torch belum siap:", e)
import numpy, pandas, pyarrow, tqdm
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| pyarrow", pyarrow.__version__, "| tqdm", tqdm.__version__)


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. Tulis source modul (verbatim dari repo) ke /content/src
import base64, os, sys

SRC_ROOT = "/content/src"
EMBEDDED = {
    'smart_extract/__init__.py': 'IiIiU21hcnQgRXh0cmFjdCBWOCBwYWNrYWdlLiIiIgoK',
    'smart_extract/contract.py': 'IiIiU2hhcmVkIFNtYXJ0IEV4dHJhY3QgVjggYmVzdC1tb2RlIGZlYXR1cmUgY29udHJhY3QuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGFyZ3BhcnNlIGltcG9ydCBOYW1lc3BhY2UKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBJdGVyYWJsZQoKaW1wb3J0IG51bXB5IGFzIG5wCgpGRUFUVVJFX1NDSEVNQSA9ICJiaXNpbmRvX3NtYXJ0X3Y4X2J0al9nbG9iYWxfbG9jYWxfMTgwX2Jlc3QiCkZFQVRVUkVfTU9ERSA9ICJidGpfZ2xvYmFsX2xvY2FsIgpGRUFUVVJFX0RJTSA9IDE4MApUQVJHRVRfRlBTID0gMTAuMApFWFRSQUNUX1BST0ZJTEUgPSAic21hcnRfdjhfYmVzdCIKClNMSUNFX1NIT1VMREVSUyA9IHNsaWNlKDAsIDYpClNMSUNFX0xFRlRfR0xPQkFMID0gc2xpY2UoNiwgMzkpClNMSUNFX1JJR0hUX0dMT0JBTCA9IHNsaWNlKDM5LCA3MikKU0xJQ0VfTEVGVF9MT0NBTCA9IHNsaWNlKDcyLCAxMDUpClNMSUNFX1JJR0hUX0xPQ0FMID0gc2xpY2UoMTA1LCAxMzgpClNMSUNFX0xFRlRfQU5HTEVTID0gc2xpY2UoMTM4LCAxNTQpClNMSUNFX1JJR0hUX0FOR0xFUyA9IHNsaWNlKDE1NCwgMTcwKQpTTElDRV9NRVRBID0gc2xpY2UoMTcwLCAxODApCgpJRFhfTUVUQV9MRUZUX1BSRVNFTlQgPSAwCklEWF9NRVRBX1JJR0hUX1BSRVNFTlQgPSAxCklEWF9NRVRBX0xFRlRfREVURUNURUQgPSAyCklEWF9NRVRBX1JJR0hUX0RFVEVDVEVEID0gMwpJRFhfTUVUQV9MRUZUX0hFTEQgPSA0CklEWF9NRVRBX1JJR0hUX0hFTEQgPSA1CklEWF9NRVRBX1NIT1VMREVSX09LID0gNgpJRFhfTUVUQV9TSE9VTERFUl9TQ0FMRSA9IDcKSURYX01FVEFfTEVGVF9TQ09SRSA9IDgKSURYX01FVEFfUklHSFRfU0NPUkUgPSA5CgpCRVNUX0ZBTExCQUNLX1ZBUklBTlRTID0gImF1dG8sY2xhaGVfc2hhcnAsZ2FtbWFfYnJpZ2h0LHNoYXJwLGRlbm9pc2VfY2xhaGVfc2hhcnAsbm9uZSIKCkJFU1RfRVhUUkFDVF9TRVRUSU5HUyA9IHsKICAgICJmZWF0dXJlX21vZGUiOiBGRUFUVVJFX01PREUsCiAgICAidGFyZ2V0X2ZwcyI6IFRBUkdFVF9GUFMsCiAgICAid2lkdGgiOiA2NDAsCiAgICAiaGVpZ2h0IjogNDgwLAogICAgImNlbnRlcl9jcm9wIjogMS4wLAogICAgInByb2Nfd2lkdGgiOiAzODQsCiAgICAic2hvdWxkZXJfYmFja2VuZCI6ICJtcC1wb3NlIiwKICAgICJwb3NlX2V2ZXJ5IjogMywKICAgICJwb3NlX3Byb2Nfd2lkdGgiOiAyNTYsCiAgICAiaGFuZF9tb2RlbF9jb21wbGV4aXR5IjogMCwKICAgICJwb3NlX21vZGVsX2NvbXBsZXhpdHkiOiAwLAogICAgImRldF9jb25mIjogMC40MCwKICAgICJ0cmFja19jb25mIjogMC40NSwKICAgICJzbW9vdGhfYWxwaGEiOiAwLjc4LAogICAgInNob3VsZGVyX3Ntb290aF9hbHBoYSI6IDAuMzUsCiAgICAiaG9sZF9mcmFtZXMiOiA1LAogICAgInNtYXJ0X21vZGUiOiAiYmVzdCIsCiAgICAic2VhcmNoX3JhZGl1cyI6IDIsCiAgICAiZW5oYW5jZSI6ICJhdXRvIiwKICAgICJmYWxsYmFja192YXJpYW50cyI6IEJFU1RfRkFMTEJBQ0tfVkFSSUFOVFMsCiAgICAiZ2lmX3dpZHRoIjogNDIwLAp9CgoKZGVmIG1ha2VfYmVzdF9hcmdzKCoqb3ZlcnJpZGVzKSAtPiBOYW1lc3BhY2U6CiAgICAiIiJSZXR1cm4gYW4gYXJncGFyc2UtY29tcGF0aWJsZSBuYW1lc3BhY2UgZm9yIHRoZSBiZXN0IGV4dHJhY3QgcHJvZmlsZS4iIiIKICAgIHZhbHVlcyA9IHsKICAgICAgICAidmlkZW8iOiBOb25lLAogICAgICAgICJiYXRjaF9kaXIiOiBOb25lLAogICAgICAgICJvdXRfZGlyIjogTm9uZSwKICAgICAgICAic2F2ZV9naWYiOiBUcnVlLAogICAgICAgICJub19naWYiOiBGYWxzZSwKICAgICAgICAic2F2ZV9tcDQiOiBGYWxzZSwKICAgICAgICAic2tlbGV0b25fYmciOiAiYmxhY2siLAogICAgICAgICJxdWlldCI6IEZhbHNlLAogICAgICAgICJtaXJyb3JfaW5wdXQiOiBGYWxzZSwKICAgICAgICAibm9fbWlycm9yX2hhbmRlZG5lc3MiOiBGYWxzZSwKICAgICAgICAid2Vha19oYW5kX3RocmVzaG9sZCI6IDEuNSwKICAgIH0KICAgIHZhbHVlcy51cGRhdGUoQkVTVF9FWFRSQUNUX1NFVFRJTkdTKQogICAgdmFsdWVzLnVwZGF0ZShvdmVycmlkZXMpCiAgICByZXR1cm4gTmFtZXNwYWNlKCoqdmFsdWVzKQoKCmRlZiBwYXJzZV9mZWF0dXJlX3ZhbHVlKHZhbHVlKSAtPiBucC5uZGFycmF5OgogICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgc3RyKToKICAgICAgICByZXR1cm4gbnAuZnJvbXN0cmluZyh2YWx1ZSwgc2VwPSIsIiwgZHR5cGU9bnAuZmxvYXQzMikKICAgIHJldHVybiBucC5hc2FycmF5KHZhbHVlLCBkdHlwZT1ucC5mbG9hdDMyKS5yZXNoYXBlKC0xKQoKCmRlZiBmb3JtYXRfZmVhdHVyZV92YWx1ZShmZWF0dXJlczogSXRlcmFibGVbZmxvYXRdKSAtPiBzdHI6CiAgICByZXR1cm4gIiwiLmpvaW4obWFwKHN0ciwgbnAuYXNhcnJheShmZWF0dXJlcywgZHR5cGU9bnAuZmxvYXQzMikucmVzaGFwZSgtMSkudG9saXN0KCkpKQoKCmRlZiBlbnN1cmVfZmVhdHVyZV9kaW0oc2VxdWVuY2UsIGV4cGVjdGVkX2RpbTogaW50ID0gRkVBVFVSRV9ESU0pIC0+IG5wLm5kYXJyYXk6CiAgICBhcnIgPSBucC5hc2FycmF5KHNlcXVlbmNlLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgaWYgYXJyLm5kaW0gPT0gMToKICAgICAgICBhcnIgPSBhcnIucmVzaGFwZSgxLCAtMSkKICAgIGlmIGFyci5uZGltICE9IDIgb3IgYXJyLnNoYXBlWzFdICE9IGludChleHBlY3RlZF9kaW0pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBmZWF0dXJlIHNoYXBlIChULCB7ZXhwZWN0ZWRfZGltfSksIGdvdCB7YXJyLnNoYXBlfSIpCiAgICBpZiBub3QgbnAuaXNmaW5pdGUoYXJyKS5hbGwoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJGZWF0dXJlIHNlcXVlbmNlIGNvbnRhaW5zIG5vbi1maW5pdGUgdmFsdWVzIikKICAgIHJldHVybiBhcnIuYXN0eXBlKG5wLmZsb2F0MzIsIGNvcHk9RmFsc2UpCgoKZGVmIGZpbHRlcl9jdXJyZW50X2ZlYXR1cmVfcm93cyhkZik6CiAgICBpZiAiZmVhdHVyZV92ZXJzaW9uIiBub3QgaW4gZGYuY29sdW1uczoKICAgICAgICByZXR1cm4gZGYuaWxvY1swOjBdLmNvcHkoKQogICAgaWYgImZlYXR1cmVfZGltIiBpbiBkZi5jb2x1bW5zOgogICAgICAgIGRpbXMgPSBkZlsiZmVhdHVyZV9kaW0iXQogICAgICAgIHRyeToKICAgICAgICAgICAgZGltcyA9IGRpbXMuYXN0eXBlKGludCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBkZWYgX2RpbSh2YWx1ZSk6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGludChmbG9hdCh2YWx1ZSkpCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgICAgIHJldHVybiAtMQogICAgICAgICAgICBkaW1zID0gZGltcy5hcHBseShfZGltKQogICAgICAgIHJldHVybiBkZlsoZGZbImZlYXR1cmVfdmVyc2lvbiJdID09IEZFQVRVUkVfU0NIRU1BKSAmIChkaW1zID09IEZFQVRVUkVfRElNKV0uY29weSgpCiAgICByZXR1cm4gZGZbZGZbImZlYXR1cmVfdmVyc2lvbiJdID09IEZFQVRVUkVfU0NIRU1BXS5jb3B5KCkKCgpkZWYgc2FtcGxlX2dpZl9wYXRocyh2b2NhYjogc3RyLCB2aWRlb19pZDogc3RyLCByb290X2Rpcjogc3RyIHwgUGF0aCwgbW9kZXM6IEl0ZXJhYmxlW3N0cl0gPSAoIm92ZXJsYXkiLCAic2tlbGV0b24iKSkgLT4gZGljdFtzdHIsIHN0cl06CiAgICBzYWZlX3ZpZGVvX2lkID0gIiIuam9pbihjIGlmIGMuaXNhbG51bSgpIG9yIGMgaW4gIi5fLSIgZWxzZSAiXyIgZm9yIGMgaW4gc3RyKHZpZGVvX2lkKSkKICAgIGJhc2UgPSBQYXRoKHJvb3RfZGlyKSAvICJhc3NldHMiIC8gImdpZnMiIC8gInNhbXBsZXMiIC8gc3RyKHZvY2FiKQogICAgcmV0dXJuIHttb2RlOiBzdHIoYmFzZSAvIGYie3NhZmVfdmlkZW9faWR9X3ttb2RlfS5naWYiKSBmb3IgbW9kZSBpbiBtb2Rlc30KCgpkZWYgbW90aW9uX3Njb3JlKHByZXY6IG5wLm5kYXJyYXkgfCBOb25lLCBjdXJyOiBucC5uZGFycmF5IHwgTm9uZSkgLT4gdHVwbGVbZmxvYXQsIGJvb2xdOgogICAgaWYgY3VyciBpcyBOb25lOgogICAgICAgIHJldHVybiAwLjAsIEZhbHNlCiAgICBjdXJyID0gbnAuYXNhcnJheShjdXJyLCBkdHlwZT1ucC5mbG9hdDMyKS5yZXNoYXBlKC0xKQogICAgaWYgY3Vyci5zaGFwZVswXSA8IEZFQVRVUkVfRElNOgogICAgICAgIHJldHVybiAwLjAsIEZhbHNlCiAgICBtZXRhID0gY3VycltTTElDRV9NRVRBXQogICAgdmlzaWJsZSA9IGJvb2wobWV0YVtJRFhfTUVUQV9MRUZUX1BSRVNFTlRdID49IDAuNSBvciBtZXRhW0lEWF9NRVRBX1JJR0hUX1BSRVNFTlRdID49IDAuNSkKICAgIGlmIHByZXYgaXMgTm9uZToKICAgICAgICByZXR1cm4gMC4wLCB2aXNpYmxlCiAgICBwcmV2ID0gbnAuYXNhcnJheShwcmV2LCBkdHlwZT1ucC5mbG9hdDMyKS5yZXNoYXBlKC0xKQogICAgaWYgcHJldi5zaGFwZVswXSA8IEZFQVRVUkVfRElNOgogICAgICAgIHJldHVybiAwLjAsIHZpc2libGUKICAgIGNodW5rcyA9IFsKICAgICAgICAoU0xJQ0VfTEVGVF9HTE9CQUwsIElEWF9NRVRBX0xFRlRfUFJFU0VOVCksCiAgICAgICAgKFNMSUNFX1JJR0hUX0dMT0JBTCwgSURYX01FVEFfUklHSFRfUFJFU0VOVCksCiAgICAgICAgKFNMSUNFX0xFRlRfTE9DQUwsIElEWF9NRVRBX0xFRlRfUFJFU0VOVCksCiAgICAgICAgKFNMSUNFX1JJR0hUX0xPQ0FMLCBJRFhfTUVUQV9SSUdIVF9QUkVTRU5UKSwKICAgIF0KICAgIHNjb3JlcyA9IFtdCiAgICBmb3Igc2wsIG1ldGFfaWR4IGluIGNodW5rczoKICAgICAgICBpZiBtZXRhW21ldGFfaWR4XSA+PSAwLjU6CiAgICAgICAgICAgIHNjb3Jlcy5hcHBlbmQoZmxvYXQobnAubGluYWxnLm5vcm0oY3VycltzbF0gLSBwcmV2W3NsXSkpKQogICAgcmV0dXJuIChmbG9hdChtYXgoc2NvcmVzKSkgaWYgc2NvcmVzIGVsc2UgMC4wKSwgdmlzaWJsZQo=',
    'feature_schemas.py': 'IiIiRmVhdHVyZSBzY2hlbWEgcmVnaXN0cnkgZm9yIEJJU0lORE8gZGF0YXNldHMgYW5kIEdSVSBjaGVja3BvaW50cy4KCnYxMCBwb2xpY3k6Ci0gYGAtLXNjaGVtYSBhbGxgYCBub3cgbWVhbnMgZXZlcnkgbWFpbnRhaW5lZCBzY2hlbWEsIGluY2x1ZGluZwogIHNtYXJ0MTgwX2ZhY2UxNTg0LiBVc2UgYGBiYXNlYGAgLyBgYG9yaWdpbmFsYGAgZm9yIHRoZSBvcmlnaW5hbCB0aHJlZSBzY2hlbWFzCiAgb25seTogc21hcnQxODAsIGtodWt1aDE2MjksIGFkaTE2NjIuCi0gS2h1a3VoIDE2MjktRCBhbmQgQWRpIDE2NjItRCBhbHJlYWR5IGluY2x1ZGUgTWVkaWFQaXBlIGZhY2UgbGFuZG1hcmtzLgotIEFkZCBvbmx5IG9uZSBleHRyYSBmYWNlIHNjaGVtYTogc21hcnQxODBfZmFjZTE1ODQgPSBzbWFydDE4MCArIGZhY2U0NjggeHl6LgogIEl0IGlzIG9wdC1pbiB2aWEgYGAtLXNjaGVtYSBzbWFydDE4MF9mYWNlMTU4NGBgIC8gYGAtLXNjaGVtYSBmYWNlYGAgLwogIGBgLS1zY2hlbWEgZnVsbGBgIHNvIG9sZCByZWNvcmRpbmcgY29tbWFuZHMgZG8gbm90IHN1ZGRlbmx5IGdldCBoZWF2aWVyLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEl0ZXJhYmxlCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSBzbWFydF9leHRyYWN0IGltcG9ydCBjb250cmFjdCBhcyBzYwoKClJPT1RfRElSID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0KREFUQVNFVF9ST09UID0gUk9PVF9ESVIgLyAiZGF0YXNldF9wYXJxdWV0cyIKTU9ERUxfUk9PVCA9IFJPT1RfRElSIC8gIm1vZGVscyIKREVGQVVMVF9TQ0hFTUEgPSAic21hcnQxODAiCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgRmVhdHVyZVNjaGVtYToKICAgIG5hbWU6IHN0cgogICAgZGlzcGxheV9uYW1lOiBzdHIKICAgIGZlYXR1cmVfc2NoZW1hOiBzdHIKICAgIGZlYXR1cmVfbW9kZTogc3RyCiAgICBmZWF0dXJlX2RpbTogaW50CiAgICBkYXRhc2V0X3N1YmRpcjogc3RyCiAgICBtb2RlbF9zdWJkaXI6IHN0cgogICAgZXh0cmFjdG9yOiBzdHIKICAgIHRhcmdldF9mcHM6IGZsb2F0ID0gMTAuMAogICAgdXNlc19mYWNlOiBib29sID0gRmFsc2UKICAgIGJhc2Vfc2NoZW1hOiBzdHIgPSAiIgogICAgbm90ZXM6IHN0ciA9ICIiCgoKQkFTRV9TQ0hFTUFfTkFNRVMgPSAoInNtYXJ0MTgwIiwgImtodWt1aDE2MjkiLCAiYWRpMTY2MiIpCkVYVFJBX1NDSEVNQV9OQU1FUyA9ICgic21hcnQxODBfZmFjZTE1ODQiLCkKRlVMTF9TQ0hFTUFfTkFNRVMgPSBCQVNFX1NDSEVNQV9OQU1FUyArIEVYVFJBX1NDSEVNQV9OQU1FUwpGQUNFX0VOQUJMRURfU0NIRU1BX05BTUVTID0gKCJzbWFydDE4MF9mYWNlMTU4NCIsICJraHVrdWgxNjI5IiwgImFkaTE2NjIiKQoKClNDSEVNQVM6IGRpY3Rbc3RyLCBGZWF0dXJlU2NoZW1hXSA9IHsKICAgICJzbWFydDE4MCI6IEZlYXR1cmVTY2hlbWEoCiAgICAgICAgbmFtZT0ic21hcnQxODAiLAogICAgICAgIGRpc3BsYXlfbmFtZT0iU21hcnQgVjggMTgwLUQgKGhhbmRzICsgc2hvdWxkZXJzLCBubyBmYWNlKSIsCiAgICAgICAgZmVhdHVyZV9zY2hlbWE9c2MuRkVBVFVSRV9TQ0hFTUEsCiAgICAgICAgZmVhdHVyZV9tb2RlPXNjLkZFQVRVUkVfTU9ERSwKICAgICAgICBmZWF0dXJlX2RpbT1zYy5GRUFUVVJFX0RJTSwKICAgICAgICBkYXRhc2V0X3N1YmRpcj0ic21hcnQxODAiLAogICAgICAgIG1vZGVsX3N1YmRpcj0ic21hcnQxODAiLAogICAgICAgIGV4dHJhY3Rvcj0ic21hcnRfdjgiLAogICAgICAgIHRhcmdldF9mcHM9c2MuVEFSR0VUX0ZQUywKICAgICAgICB1c2VzX2ZhY2U9RmFsc2UsCiAgICAgICAgYmFzZV9zY2hlbWE9InNtYXJ0MTgwIiwKICAgICAgICBub3Rlcz0iRmFzdCBjb21wYWN0IFNtYXJ0IFY4IGZlYXR1cmU6IHNob3VsZGVycyArIHNlbGVjdGVkIGhhbmQgcG9pbnRzICsgbG9jYWwgZ2VvbWV0cnkgKyBhbmdsZXMgKyBtZXRhZGF0YS4iLAogICAgKSwKICAgICJraHVrdWgxNjI5IjogRmVhdHVyZVNjaGVtYSgKICAgICAgICBuYW1lPSJraHVrdWgxNjI5IiwKICAgICAgICBkaXNwbGF5X25hbWU9IktodWt1aCBIb2xpc3RpYyAxNjI5LUQgKGluY2x1ZGVzIGZhY2UpIiwKICAgICAgICBmZWF0dXJlX3NjaGVtYT0iYmlzaW5kb19raHVrdWhfaG9saXN0aWNfMTYyOV8xMGZwcyIsCiAgICAgICAgZmVhdHVyZV9tb2RlPSJraHVrdWhfaG9saXN0aWNfMTYyOSIsCiAgICAgICAgZmVhdHVyZV9kaW09MTYyOSwKICAgICAgICBkYXRhc2V0X3N1YmRpcj0ia2h1a3VoMTYyOSIsCiAgICAgICAgbW9kZWxfc3ViZGlyPSJraHVrdWgxNjI5IiwKICAgICAgICBleHRyYWN0b3I9ImhvbGlzdGljIiwKICAgICAgICB0YXJnZXRfZnBzPTEwLjAsCiAgICAgICAgdXNlc19mYWNlPVRydWUsCiAgICAgICAgYmFzZV9zY2hlbWE9ImtodWt1aDE2MjkiLAogICAgICAgIG5vdGVzPSJSaWdodCBoYW5kIHh5eiArIGxlZnQgaGFuZCB4eXogKyBwb3NlIHh5eiArIGZhY2UgeHl6LiBGYWNlIGlzIGFscmVhZHkgcGFydCBvZiB0aGUgc2NoZW1hLiIsCiAgICApLAogICAgImFkaTE2NjIiOiBGZWF0dXJlU2NoZW1hKAogICAgICAgIG5hbWU9ImFkaTE2NjIiLAogICAgICAgIGRpc3BsYXlfbmFtZT0iQWRpIEhvbGlzdGljIDE2NjItRCAoaW5jbHVkZXMgZmFjZSkiLAogICAgICAgIGZlYXR1cmVfc2NoZW1hPSJiaXNpbmRvX2FkaV9ob2xpc3RpY18xNjYyXzEwZnBzIiwKICAgICAgICBmZWF0dXJlX21vZGU9ImFkaV9ob2xpc3RpY18xNjYyIiwKICAgICAgICBmZWF0dXJlX2RpbT0xNjYyLAogICAgICAgIGRhdGFzZXRfc3ViZGlyPSJhZGkxNjYyIiwKICAgICAgICBtb2RlbF9zdWJkaXI9ImFkaTE2NjIiLAogICAgICAgIGV4dHJhY3Rvcj0iaG9saXN0aWMiLAogICAgICAgIHRhcmdldF9mcHM9MTAuMCwKICAgICAgICB1c2VzX2ZhY2U9VHJ1ZSwKICAgICAgICBiYXNlX3NjaGVtYT0iYWRpMTY2MiIsCiAgICAgICAgbm90ZXM9IlBvc2UgeHl6dyArIGZhY2UgeHl6ICsgbGVmdCBoYW5kIHh5eiArIHJpZ2h0IGhhbmQgeHl6LiBGYWNlIGlzIGFscmVhZHkgcGFydCBvZiB0aGUgc2NoZW1hLiIsCiAgICApLAogICAgInNtYXJ0MTgwX2ZhY2UxNTg0IjogRmVhdHVyZVNjaGVtYSgKICAgICAgICBuYW1lPSJzbWFydDE4MF9mYWNlMTU4NCIsCiAgICAgICAgZGlzcGxheV9uYW1lPSJTbWFydCBWOCAxODAtRCArIEZhY2UgMTQwNC1EID0gMTU4NC1EIiwKICAgICAgICBmZWF0dXJlX3NjaGVtYT0iYmlzaW5kb19zbWFydF92OF8xODBfcGx1c19mYWNlNDY4eHl6XzE1ODRfMTBmcHMiLAogICAgICAgIGZlYXR1cmVfbW9kZT0ic21hcnQxODBfcGx1c19mYWNlNDY4eHl6IiwKICAgICAgICBmZWF0dXJlX2RpbT0xNTg0LAogICAgICAgIGRhdGFzZXRfc3ViZGlyPSJzbWFydDE4MF9mYWNlMTU4NCIsCiAgICAgICAgbW9kZWxfc3ViZGlyPSJzbWFydDE4MF9mYWNlMTU4NCIsCiAgICAgICAgZXh0cmFjdG9yPSJob2xpc3RpYyIsCiAgICAgICAgdGFyZ2V0X2Zwcz0xMC4wLAogICAgICAgIHVzZXNfZmFjZT1UcnVlLAogICAgICAgIGJhc2Vfc2NoZW1hPSJzbWFydDE4MCIsCiAgICAgICAgbm90ZXM9Ik9wdC1pbiBmZWF0dXJlOiBTbWFydDE4MC1jb21wYXRpYmxlIGNvbXBhY3QgaGFuZC9zaG91bGRlciB2ZWN0b3IgcGx1cyBNZWRpYVBpcGUgZmFjZSB4eXogbGFuZG1hcmtzLiIsCiAgICApLAp9ClNDSEVNQV9OQU1FUyA9IHR1cGxlKFNDSEVNQVMua2V5cygpKQoKClNNQVJUMTgwX1NFTEVDVEVEX1BPSU5UUyA9ICgKICAgICJ3cmlzdCIsCiAgICAicGFsbV9jZW50ZXIiLAogICAgInRodW1iX3RpcCIsCiAgICAiaW5kZXhfbWNwIiwKICAgICJpbmRleF90aXAiLAogICAgIm1pZGRsZV9tY3AiLAogICAgIm1pZGRsZV90aXAiLAogICAgInJpbmdfbWNwIiwKICAgICJyaW5nX3RpcCIsCiAgICAicGlua3lfbWNwIiwKICAgICJwaW5reV90aXAiLAopClNNQVJUMTgwX01FVEFfTkFNRVMgPSAoCiAgICAibGVmdF9wcmVzZW50IiwKICAgICJyaWdodF9wcmVzZW50IiwKICAgICJsZWZ0X2RldGVjdGVkIiwKICAgICJyaWdodF9kZXRlY3RlZCIsCiAgICAibGVmdF9oZWxkIiwKICAgICJyaWdodF9oZWxkIiwKICAgICJzaG91bGRlcl9vayIsCiAgICAic2hvdWxkZXJfc2NhbGUiLAogICAgImxlZnRfc2NvcmUiLAogICAgInJpZ2h0X3Njb3JlIiwKKQpBTkdMRV9OQU1FUyA9ICgKICAgICJ0aHVtYl9jbWMiLAogICAgInRodW1iX21jcCIsCiAgICAidGh1bWJfaXAiLAogICAgInRodW1iX3RpcF9jaGFpbiIsCiAgICAiaW5kZXhfbWNwIiwKICAgICJpbmRleF9waXAiLAogICAgImluZGV4X2RpcCIsCiAgICAibWlkZGxlX21jcCIsCiAgICAibWlkZGxlX3BpcCIsCiAgICAibWlkZGxlX2RpcCIsCiAgICAicmluZ19tY3AiLAogICAgInJpbmdfcGlwIiwKICAgICJyaW5nX2RpcCIsCiAgICAicGlua3lfbWNwIiwKICAgICJwaW5reV9waXAiLAogICAgInBpbmt5X2RpcCIsCikKCgpkZWYgbm9ybWFsaXplX3NjaGVtYV9uYW1lKHNjaGVtYTogc3RyIHwgTm9uZSA9IE5vbmUpIC0+IHN0cjoKICAgIHZhbHVlID0gc3RyKHNjaGVtYSBvciBERUZBVUxUX1NDSEVNQSkuc3RyaXAoKS5sb3dlcigpLnJlcGxhY2UoIi0iLCAiXyIpCiAgICBhbGlhc2VzID0gewogICAgICAgICJkZWZhdWx0IjogREVGQVVMVF9TQ0hFTUEsCiAgICAgICAgInNtYXJ0IjogInNtYXJ0MTgwIiwKICAgICAgICAidjgiOiAic21hcnQxODAiLAogICAgICAgICIxODAiOiAic21hcnQxODAiLAogICAgICAgICJraHVrdWgiOiAia2h1a3VoMTYyOSIsCiAgICAgICAgImtodWt1aF9mYWNlIjogImtodWt1aDE2MjkiLAogICAgICAgICJraHVrdWgxNjI5X2ZhY2UiOiAia2h1a3VoMTYyOSIsCiAgICAgICAgIjE2MjkiOiAia2h1a3VoMTYyOSIsCiAgICAgICAgIjE2MjlfZmFjZSI6ICJraHVrdWgxNjI5IiwKICAgICAgICAiYWRpIjogImFkaTE2NjIiLAogICAgICAgICJhZGhpIjogImFkaTE2NjIiLAogICAgICAgICJhZGlfZmFjZSI6ICJhZGkxNjYyIiwKICAgICAgICAiYWRoaV9mYWNlIjogImFkaTE2NjIiLAogICAgICAgICJhZGkxNjYyX2ZhY2UiOiAiYWRpMTY2MiIsCiAgICAgICAgIjE2NjIiOiAiYWRpMTY2MiIsCiAgICAgICAgIjE2NjJfZmFjZSI6ICJhZGkxNjYyIiwKICAgICAgICAic21hcnRfZmFjZSI6ICJzbWFydDE4MF9mYWNlMTU4NCIsCiAgICAgICAgInNtYXJ0MTgwX2ZhY2UiOiAic21hcnQxODBfZmFjZTE1ODQiLAogICAgICAgICJzbWFydDE4MGZhY2UiOiAic21hcnQxODBfZmFjZTE1ODQiLAogICAgICAgICIxODBfZmFjZSI6ICJzbWFydDE4MF9mYWNlMTU4NCIsCiAgICAgICAgIjE4MGZhY2UiOiAic21hcnQxODBfZmFjZTE1ODQiLAogICAgICAgICJmYWNlMTgwIjogInNtYXJ0MTgwX2ZhY2UxNTg0IiwKICAgICAgICAiY29tcGFjdF9mYWNlIjogInNtYXJ0MTgwX2ZhY2UxNTg0IiwKICAgIH0KICAgIHZhbHVlID0gYWxpYXNlcy5nZXQodmFsdWUsIHZhbHVlKQogICAgaWYgdmFsdWUgbm90IGluIFNDSEVNQVM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gZmVhdHVyZSBzY2hlbWEgJ3tzY2hlbWF9Jy4gUGlsaWg6IHsnLCAnLmpvaW4oU0NIRU1BX05BTUVTKX0iKQogICAgcmV0dXJuIHZhbHVlCgoKZGVmIGV4cGFuZF9zY2hlbWFfbmFtZXMoc2NoZW1hOiBzdHIgfCBOb25lID0gTm9uZSkgLT4gdHVwbGVbc3RyLCAuLi5dOgogICAgdmFsdWUgPSBzdHIoc2NoZW1hIG9yIERFRkFVTFRfU0NIRU1BKS5zdHJpcCgpLmxvd2VyKCkucmVwbGFjZSgiLSIsICJfIikKICAgIGlmIHZhbHVlIGluIHsiYmFzZSIsICJvcmlnaW5hbCIsICJvcmlnaW5hbHMiLCAiYXNsaSIsICJrZXRpZ2FueWEifToKICAgICAgICByZXR1cm4gQkFTRV9TQ0hFTUFfTkFNRVMKICAgICMgZmFjZSA9IG9ubHkgdGhlIG5ldyBvcHRpb25hbCAxODArZmFjZSBzY2hlbWEuICBBZGkvS2h1a3VoIGFscmVhZHkgY29udGFpbiBmYWNlLgogICAgaWYgdmFsdWUgaW4geyJmYWNlIiwgImZhY2VzIiwgIndhamFoIiwgImV4dHJhIiwgImV4dHJhcyIsICJuZXciLCAidGFtYmFoYW4ifToKICAgICAgICByZXR1cm4gRVhUUkFfU0NIRU1BX05BTUVTCiAgICAjIGFsbC9mdWxsIGV4cGxpY2l0bHkgaW5jbHVkZXMgYWxsIGZvdXIgc2NoZW1hcy4KICAgIGlmIHZhbHVlIGluIHsiYWxsIiwgImZ1bGwiLCAiYWxsX2ZhY2UiLCAiYWxsX3dpdGhfZmFjZSIsICJzZW11YSIsICJzZW11YV9wbHVzX2ZhY2UiLCAiYmFzZV9wbHVzX2ZhY2UifToKICAgICAgICByZXR1cm4gRlVMTF9TQ0hFTUFfTkFNRVMKICAgIHJldHVybiAobm9ybWFsaXplX3NjaGVtYV9uYW1lKHZhbHVlKSwpCgoKZGVmIGdldF9zY2hlbWEoc2NoZW1hOiBzdHIgfCBGZWF0dXJlU2NoZW1hIHwgTm9uZSA9IE5vbmUpIC0+IEZlYXR1cmVTY2hlbWE6CiAgICBpZiBpc2luc3RhbmNlKHNjaGVtYSwgRmVhdHVyZVNjaGVtYSk6CiAgICAgICAgcmV0dXJuIHNjaGVtYQogICAgcmV0dXJuIFNDSEVNQVNbbm9ybWFsaXplX3NjaGVtYV9uYW1lKHNjaGVtYSldCgoKZGVmIGRhdGFzZXRfZGlyX2ZvcihzY2hlbWE6IHN0ciB8IEZlYXR1cmVTY2hlbWEgfCBOb25lID0gTm9uZSwgZGF0YXNldF9yb290OiBzdHIgfCBQYXRoID0gREFUQVNFVF9ST09UKSAtPiBQYXRoOgogICAgc3BlYyA9IGdldF9zY2hlbWEoc2NoZW1hKQogICAgcm9vdCA9IFBhdGgoZGF0YXNldF9yb290KQogICAgaWYgcm9vdC5uYW1lID09IHNwZWMuZGF0YXNldF9zdWJkaXI6CiAgICAgICAgcmV0dXJuIHJvb3QKICAgIHJldHVybiByb290IC8gc3BlYy5kYXRhc2V0X3N1YmRpcgoKCmRlZiBkYXRhc2V0X3BhcnF1ZXRfcGF0aHMoCiAgICBzY2hlbWE6IHN0ciB8IEZlYXR1cmVTY2hlbWEgfCBOb25lID0gTm9uZSwKICAgIGRhdGFzZXRfcm9vdDogc3RyIHwgUGF0aCA9IERBVEFTRVRfUk9PVCwKICAgIGluY2x1ZGVfbGVnYWN5X3NtYXJ0MTgwOiBib29sID0gVHJ1ZSwKKSAtPiBsaXN0W1BhdGhdOgogICAgc3BlYyA9IGdldF9zY2hlbWEoc2NoZW1hKQogICAgcm9vdCA9IFBhdGgoZGF0YXNldF9yb290KQogICAgcGF0aHM6IGxpc3RbUGF0aF0gPSBbXQogICAgc2NoZW1hX2RpciA9IGRhdGFzZXRfZGlyX2ZvcihzcGVjLCByb290KQogICAgaWYgc2NoZW1hX2Rpci5leGlzdHMoKToKICAgICAgICBwYXRocy5leHRlbmQoc29ydGVkKHNjaGVtYV9kaXIuZ2xvYigiKi5wYXJxdWV0IikpKQogICAgZGlyZWN0X3BhdGhzID0gc29ydGVkKHJvb3QuZ2xvYigiKi5wYXJxdWV0IikpIGlmIHJvb3QuZXhpc3RzKCkgZWxzZSBbXQogICAgaWYgcm9vdC5uYW1lID09IHNwZWMuZGF0YXNldF9zdWJkaXI6CiAgICAgICAgcGF0aHMuZXh0ZW5kKHBhdGggZm9yIHBhdGggaW4gZGlyZWN0X3BhdGhzIGlmIHBhdGggbm90IGluIHBhdGhzKQogICAgZWxpZiBzcGVjLm5hbWUgPT0gInNtYXJ0MTgwIiBhbmQgaW5jbHVkZV9sZWdhY3lfc21hcnQxODA6CiAgICAgICAgcGF0aHMuZXh0ZW5kKHBhdGggZm9yIHBhdGggaW4gZGlyZWN0X3BhdGhzIGlmIHBhdGggbm90IGluIHBhdGhzKQogICAgZWxpZiBub3QgcGF0aHM6CiAgICAgICAgcGF0aHMuZXh0ZW5kKGRpcmVjdF9wYXRocykKICAgIHJldHVybiBwYXRocwoKCmRlZiBtb2RlbF9kaXJfZm9yKHNjaGVtYTogc3RyIHwgRmVhdHVyZVNjaGVtYSB8IE5vbmUgPSBOb25lLCBtb2RlbF9yb290OiBzdHIgfCBQYXRoID0gTU9ERUxfUk9PVCkgLT4gUGF0aDoKICAgIHNwZWMgPSBnZXRfc2NoZW1hKHNjaGVtYSkKICAgIHJvb3QgPSBQYXRoKG1vZGVsX3Jvb3QpCiAgICBpZiByb290Lm5hbWUgPT0gc3BlYy5tb2RlbF9zdWJkaXI6CiAgICAgICAgcmV0dXJuIHJvb3QKICAgIGlmIHJvb3QubmFtZSA9PSAiZ3J1IjoKICAgICAgICByZXR1cm4gcm9vdCAvIHNwZWMubW9kZWxfc3ViZGlyCiAgICByZXR1cm4gcm9vdCAvICJncnUiIC8gc3BlYy5tb2RlbF9zdWJkaXIKCgpkZWYgcGFyc2VfZmVhdHVyZV92YWx1ZSh2YWx1ZSkgLT4gbnAubmRhcnJheToKICAgIHJldHVybiBzYy5wYXJzZV9mZWF0dXJlX3ZhbHVlKHZhbHVlKQoKCmRlZiBmb3JtYXRfZmVhdHVyZV92YWx1ZShmZWF0dXJlczogSXRlcmFibGVbZmxvYXRdKSAtPiBzdHI6CiAgICByZXR1cm4gc2MuZm9ybWF0X2ZlYXR1cmVfdmFsdWUoZmVhdHVyZXMpCgoKZGVmIGVuc3VyZV9mZWF0dXJlX2RpbShzZXF1ZW5jZSwgc2NoZW1hOiBzdHIgfCBGZWF0dXJlU2NoZW1hIHwgaW50IHwgTm9uZSA9IE5vbmUpIC0+IG5wLm5kYXJyYXk6CiAgICBleHBlY3RlZF9kaW0gPSBpbnQoc2NoZW1hIGlmIGlzaW5zdGFuY2Uoc2NoZW1hLCBpbnQpIGVsc2UgZ2V0X3NjaGVtYShzY2hlbWEpLmZlYXR1cmVfZGltKQogICAgcmV0dXJuIHNjLmVuc3VyZV9mZWF0dXJlX2RpbShzZXF1ZW5jZSwgZXhwZWN0ZWRfZGltKQoKCmRlZiBmaWx0ZXJfZmVhdHVyZV9yb3dzKGRmOiBwZC5EYXRhRnJhbWUsIHNjaGVtYTogc3RyIHwgRmVhdHVyZVNjaGVtYSB8IE5vbmUgPSBOb25lKSAtPiBwZC5EYXRhRnJhbWU6CiAgICBzcGVjID0gZ2V0X3NjaGVtYShzY2hlbWEpCiAgICBpZiAiZmVhdHVyZV92ZXJzaW9uIiBub3QgaW4gZGYuY29sdW1uczoKICAgICAgICByZXR1cm4gZGYuaWxvY1swOjBdLmNvcHkoKQogICAgaWYgImZlYXR1cmVfZGltIiBpbiBkZi5jb2x1bW5zOgogICAgICAgIGRpbXMgPSBkZlsiZmVhdHVyZV9kaW0iXQogICAgICAgIHRyeToKICAgICAgICAgICAgZGltcyA9IGRpbXMuYXN0eXBlKGludCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBkZWYgX2RpbSh2YWx1ZSk6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGludChmbG9hdCh2YWx1ZSkpCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgICAgIHJldHVybiAtMQogICAgICAgICAgICBkaW1zID0gZGltcy5hcHBseShfZGltKQogICAgICAgIHJldHVybiBkZlsoZGZbImZlYXR1cmVfdmVyc2lvbiJdID09IHNwZWMuZmVhdHVyZV9zY2hlbWEpICYgKGRpbXMgPT0gc3BlYy5mZWF0dXJlX2RpbSldLmNvcHkoKQogICAgcmV0dXJuIGRmW2RmWyJmZWF0dXJlX3ZlcnNpb24iXSA9PSBzcGVjLmZlYXR1cmVfc2NoZW1hXS5jb3B5KCkKCgpkZWYgX2F4aXNfbmFtZXMocHJlZml4OiBzdHIsIGNvdW50OiBpbnQsIGF4ZXM6IHR1cGxlW3N0ciwgLi4uXSkgLT4gbGlzdFtzdHJdOgogICAgcmV0dXJuIFtmIntwcmVmaXh9X3tpZHg6MDNkfV97YXhpc30iIGZvciBpZHggaW4gcmFuZ2UoY291bnQpIGZvciBheGlzIGluIGF4ZXNdCgoKZGVmIHNtYXJ0MTgwX2ZlYXR1cmVfY29sdW1uX25hbWVzKCkgLT4gbGlzdFtzdHJdOgogICAgbmFtZXM6IGxpc3Rbc3RyXSA9IFtdCiAgICBuYW1lcy5leHRlbmQoW2Yic2hvdWxkZXJfe3NpZGV9X3theGlzfSIgZm9yIHNpZGUgaW4gKCJsZWZ0IiwgInJpZ2h0IikgZm9yIGF4aXMgaW4gKCJ4IiwgInkiLCAieiIpXSkKICAgIGZvciBoYW5kIGluICgibGVmdCIsICJyaWdodCIpOgogICAgICAgIGZvciBwb2ludCBpbiBTTUFSVDE4MF9TRUxFQ1RFRF9QT0lOVFM6CiAgICAgICAgICAgIGZvciBheGlzIGluICgiZ2xvYmFsX3giLCAiZ2xvYmFsX3kiLCAiZ2xvYmFsX3oiKToKICAgICAgICAgICAgICAgIG5hbWVzLmFwcGVuZChmIntoYW5kfV97cG9pbnR9X3theGlzfSIpCiAgICBmb3IgaGFuZCBpbiAoImxlZnQiLCAicmlnaHQiKToKICAgICAgICBmb3IgcG9pbnQgaW4gU01BUlQxODBfU0VMRUNURURfUE9JTlRTOgogICAgICAgICAgICBmb3IgYXhpcyBpbiAoImxvY2FsX3giLCAibG9jYWxfeSIsICJsb2NhbF96Iik6CiAgICAgICAgICAgICAgICBuYW1lcy5hcHBlbmQoZiJ7aGFuZH1fe3BvaW50fV97YXhpc30iKQogICAgZm9yIGhhbmQgaW4gKCJsZWZ0IiwgInJpZ2h0Iik6CiAgICAgICAgbmFtZXMuZXh0ZW5kKFtmIntoYW5kfV9hbmdsZV97bmFtZX0iIGZvciBuYW1lIGluIEFOR0xFX05BTUVTXSkKICAgIG5hbWVzLmV4dGVuZChbZiJtZXRhX3tuYW1lfSIgZm9yIG5hbWUgaW4gU01BUlQxODBfTUVUQV9OQU1FU10pCiAgICByZXR1cm4gbmFtZXMKCgpkZWYgZmFjZV9mZWF0dXJlX2NvbHVtbl9uYW1lcyhwcmVmaXg6IHN0ciA9ICJmYWNlIikgLT4gbGlzdFtzdHJdOgogICAgcmV0dXJuIF9heGlzX25hbWVzKHByZWZpeCwgNDY4LCAoIngiLCAieSIsICJ6IikpCgoKZGVmIGZlYXR1cmVfY29sdW1uX25hbWVzKHNjaGVtYTogc3RyIHwgRmVhdHVyZVNjaGVtYSB8IE5vbmUgPSBOb25lKSAtPiBsaXN0W3N0cl06CiAgICBzcGVjID0gZ2V0X3NjaGVtYShzY2hlbWEpCiAgICBpZiBzcGVjLm5hbWUgPT0gInNtYXJ0MTgwIjoKICAgICAgICByZXR1cm4gc21hcnQxODBfZmVhdHVyZV9jb2x1bW5fbmFtZXMoKQogICAgaWYgc3BlYy5uYW1lID09ICJzbWFydDE4MF9mYWNlMTU4NCI6CiAgICAgICAgcmV0dXJuIHNtYXJ0MTgwX2ZlYXR1cmVfY29sdW1uX25hbWVzKCkgKyBmYWNlX2ZlYXR1cmVfY29sdW1uX25hbWVzKCkKICAgIGlmIHNwZWMubmFtZSA9PSAia2h1a3VoMTYyOSI6CiAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgX2F4aXNfbmFtZXMoInJpZ2h0X2hhbmQiLCAyMSwgKCJ4IiwgInkiLCAieiIpKQogICAgICAgICAgICArIF9heGlzX25hbWVzKCJsZWZ0X2hhbmQiLCAyMSwgKCJ4IiwgInkiLCAieiIpKQogICAgICAgICAgICArIF9heGlzX25hbWVzKCJwb3NlIiwgMzMsICgieCIsICJ5IiwgInoiKSkKICAgICAgICAgICAgKyBmYWNlX2ZlYXR1cmVfY29sdW1uX25hbWVzKCkKICAgICAgICApCiAgICBpZiBzcGVjLm5hbWUgPT0gImFkaTE2NjIiOgogICAgICAgIHJldHVybiAoCiAgICAgICAgICAgIF9heGlzX25hbWVzKCJwb3NlIiwgMzMsICgieCIsICJ5IiwgInoiLCAidmlzaWJpbGl0eSIpKQogICAgICAgICAgICArIGZhY2VfZmVhdHVyZV9jb2x1bW5fbmFtZXMoKQogICAgICAgICAgICArIF9heGlzX25hbWVzKCJsZWZ0X2hhbmQiLCAyMSwgKCJ4IiwgInkiLCAieiIpKQogICAgICAgICAgICArIF9heGlzX25hbWVzKCJyaWdodF9oYW5kIiwgMjEsICgieCIsICJ5IiwgInoiKSkKICAgICAgICApCiAgICByZXR1cm4gW2YiZntpZHh9IiBmb3IgaWR4IGluIHJhbmdlKHNwZWMuZmVhdHVyZV9kaW0pXQoKCmRlZiBmdWxsX3Jhd19mZWF0dXJlX2NvbHVtbl9uYW1lcygpIC0+IGxpc3Rbc3RyXToKICAgIHJldHVybiAoCiAgICAgICAgX2F4aXNfbmFtZXMoInJpZ2h0X2hhbmQiLCAyMSwgKCJ4IiwgInkiLCAieiIpKQogICAgICAgICsgX2F4aXNfbmFtZXMoImxlZnRfaGFuZCIsIDIxLCAoIngiLCAieSIsICJ6IikpCiAgICAgICAgKyBfYXhpc19uYW1lcygicG9zZSIsIDMzLCAoIngiLCAieSIsICJ6IiwgInZpc2liaWxpdHkiKSkKICAgICAgICArIGZhY2VfZmVhdHVyZV9jb2x1bW5fbmFtZXMoKQogICAgICAgICsgW2Yic2hvdWxkZXJfe3NpZGV9X3theGlzfSIgZm9yIHNpZGUgaW4gKCJsZWZ0IiwgInJpZ2h0IikgZm9yIGF4aXMgaW4gKCJ4IiwgInkiLCAieiIsICJ2aXNpYmlsaXR5IildCiAgICApCgoKZGVmIHNhbXBsZV9naWZfcGF0aHMoCiAgICBzY2hlbWE6IHN0ciB8IEZlYXR1cmVTY2hlbWEsCiAgICB2b2NhYjogc3RyLAogICAgdmlkZW9faWQ6IHN0ciwKICAgIHJvb3RfZGlyOiBzdHIgfCBQYXRoLAogICAgbW9kZXM6IEl0ZXJhYmxlW3N0cl0gPSAoIm92ZXJsYXkiLCAic2tlbGV0b24iKSwKKSAtPiBkaWN0W3N0ciwgc3RyXToKICAgIHNwZWMgPSBnZXRfc2NoZW1hKHNjaGVtYSkKICAgIHNhZmVfdmlkZW9faWQgPSAiIi5qb2luKGMgaWYgYy5pc2FsbnVtKCkgb3IgYyBpbiAiLl8tIiBlbHNlICJfIiBmb3IgYyBpbiBzdHIodmlkZW9faWQpKQogICAgYmFzZSA9IFBhdGgocm9vdF9kaXIpIC8gImFzc2V0cyIgLyAiZ2lmcyIgLyAic2FtcGxlcyIgLyBzcGVjLm5hbWUgLyBzdHIodm9jYWIpCiAgICByZXR1cm4ge21vZGU6IHN0cihiYXNlIC8gZiJ7c2FmZV92aWRlb19pZH1fe21vZGV9LmdpZiIpIGZvciBtb2RlIGluIG1vZGVzfQoKCmRlZiBwcmVzZW5jZV9mcm9tX3ZlY3RvcihzY2hlbWE6IHN0ciB8IEZlYXR1cmVTY2hlbWEsIHZlY3RvcjogbnAubmRhcnJheSkgLT4gZGljdFtzdHIsIGZsb2F0IHwgYm9vbF06CiAgICBzcGVjID0gZ2V0X3NjaGVtYShzY2hlbWEpCiAgICB2ZWMgPSBucC5hc2FycmF5KHZlY3RvciwgZHR5cGU9bnAuZmxvYXQzMikucmVzaGFwZSgtMSkKICAgIGlmIHZlYy5zaGFwZVswXSA8IHNwZWMuZmVhdHVyZV9kaW06CiAgICAgICAgcmV0dXJuIHsibGVmdF9wcmVzZW50IjogMC4wLCAicmlnaHRfcHJlc2VudCI6IDAuMCwgInNob3VsZGVyX29rIjogRmFsc2UsICJ2aXNpYmxlIjogRmFsc2V9CgogICAgaWYgc3BlYy5uYW1lIGluIHsic21hcnQxODAiLCAic21hcnQxODBfZmFjZTE1ODQifToKICAgICAgICBtZXRhID0gdmVjW3NjLlNMSUNFX01FVEFdCiAgICAgICAgbGVmdCA9IGZsb2F0KG1ldGFbc2MuSURYX01FVEFfTEVGVF9QUkVTRU5UXSkKICAgICAgICByaWdodCA9IGZsb2F0KG1ldGFbc2MuSURYX01FVEFfUklHSFRfUFJFU0VOVF0pCiAgICAgICAgc2hvdWxkZXIgPSBib29sKG1ldGFbc2MuSURYX01FVEFfU0hPVUxERVJfT0tdID49IDAuNSkKICAgIGVsaWYgc3BlYy5uYW1lID09ICJraHVrdWgxNjI5IjoKICAgICAgICByaWdodF9jaHVuayA9IHZlY1swOjYzXQogICAgICAgIGxlZnRfY2h1bmsgPSB2ZWNbNjM6MTI2XQogICAgICAgIHBvc2VfY2h1bmsgPSB2ZWNbMTI2OjIyNV0KICAgICAgICBsZWZ0ID0gZmxvYXQobnAubGluYWxnLm5vcm0obGVmdF9jaHVuaykgPiAxZS02KQogICAgICAgIHJpZ2h0ID0gZmxvYXQobnAubGluYWxnLm5vcm0ocmlnaHRfY2h1bmspID4gMWUtNikKICAgICAgICBzaG91bGRlciA9IGJvb2wobnAubGluYWxnLm5vcm0ocG9zZV9jaHVuaykgPiAxZS02KQogICAgZWxzZToKICAgICAgICBwb3NlX2NodW5rID0gdmVjWzA6MTMyXQogICAgICAgIGxlZnRfY2h1bmsgPSB2ZWNbMTUzNjoxNTk5XQogICAgICAgIHJpZ2h0X2NodW5rID0gdmVjWzE1OTk6MTY2Ml0KICAgICAgICBsZWZ0ID0gZmxvYXQobnAubGluYWxnLm5vcm0obGVmdF9jaHVuaykgPiAxZS02KQogICAgICAgIHJpZ2h0ID0gZmxvYXQobnAubGluYWxnLm5vcm0ocmlnaHRfY2h1bmspID4gMWUtNikKICAgICAgICBzaG91bGRlciA9IGJvb2wobnAubGluYWxnLm5vcm0ocG9zZV9jaHVuaykgPiAxZS02KQoKICAgIHJldHVybiB7CiAgICAgICAgImxlZnRfcHJlc2VudCI6IGxlZnQsCiAgICAgICAgInJpZ2h0X3ByZXNlbnQiOiByaWdodCwKICAgICAgICAic2hvdWxkZXJfb2siOiBzaG91bGRlciwKICAgICAgICAidmlzaWJsZSI6IGJvb2wobGVmdCA+PSAwLjUgb3IgcmlnaHQgPj0gMC41KSwKICAgIH0KCgpkZWYgbW90aW9uX3Njb3JlKAogICAgc2NoZW1hOiBzdHIgfCBGZWF0dXJlU2NoZW1hLAogICAgcHJldjogbnAubmRhcnJheSB8IE5vbmUsCiAgICBjdXJyOiBucC5uZGFycmF5IHwgTm9uZSwKKSAtPiB0dXBsZVtmbG9hdCwgYm9vbF06CiAgICBzcGVjID0gZ2V0X3NjaGVtYShzY2hlbWEpCiAgICBpZiBjdXJyIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIDAuMCwgRmFsc2UKICAgIGlmIHNwZWMubmFtZSBpbiB7InNtYXJ0MTgwIiwgInNtYXJ0MTgwX2ZhY2UxNTg0In06CiAgICAgICAgcmV0dXJuIHNjLm1vdGlvbl9zY29yZShwcmV2LCBjdXJyKQoKICAgIGN1cnJfdmVjID0gZW5zdXJlX2ZlYXR1cmVfZGltKGN1cnIsIHNwZWMpWzBdCiAgICBwcmVzZW50ID0gcHJlc2VuY2VfZnJvbV92ZWN0b3Ioc3BlYywgY3Vycl92ZWMpCiAgICB2aXNpYmxlID0gYm9vbChwcmVzZW50WyJ2aXNpYmxlIl0pCiAgICBpZiBwcmV2IGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIDAuMCwgdmlzaWJsZQogICAgcHJldl92ZWMgPSBlbnN1cmVfZmVhdHVyZV9kaW0ocHJldiwgc3BlYylbMF0KICAgIGlmIHNwZWMubmFtZSA9PSAia2h1a3VoMTYyOSI6CiAgICAgICAgY2h1bmtzID0gWyg2MywgMTI2KSwgKDAsIDYzKV0KICAgIGVsc2U6CiAgICAgICAgY2h1bmtzID0gWygxNTM2LCAxNTk5KSwgKDE1OTksIDE2NjIpXQogICAgc2NvcmVzID0gW10KICAgIGZvciBzdGFydCwgc3RvcCBpbiBjaHVua3M6CiAgICAgICAgaWYgbnAubGluYWxnLm5vcm0oY3Vycl92ZWNbc3RhcnQ6c3RvcF0pID4gMWUtNjoKICAgICAgICAgICAgc2NvcmVzLmFwcGVuZChmbG9hdChucC5saW5hbGcubm9ybShjdXJyX3ZlY1tzdGFydDpzdG9wXSAtIHByZXZfdmVjW3N0YXJ0OnN0b3BdKSAvIG5wLnNxcnQoc3RvcCAtIHN0YXJ0KSkpCiAgICByZXR1cm4gKGZsb2F0KG1heChzY29yZXMpKSBpZiBzY29yZXMgZWxzZSAwLjApLCB2aXNpYmxlCg==',
    'gru_adi.py': 'IiIiQWRpLXN0eWxlIEdSVSBhcmNoaXRlY3R1cmUgd2l0aCBSZUxVIGNhbmRpZGF0ZSBhY3RpdmF0aW9uLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG1hdGgKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgoKClRBUkdFVF9GUkFNRVMgPSA2MApERUZBVUxUX0xSID0gMWUtMwpERUZBVUxUX0JBVENIX1NJWkUgPSAzMgpERUZBVUxUX0VQT0NIUyA9IDMwMApERUZBVUxUX0RST1BPVVQgPSAwLjMwCkRFRkFVTFRfUEFUSUVOQ0UgPSA0MAoKCmNsYXNzIFJlTFVHUlVDZWxsKG5uLk1vZHVsZSk6CiAgICAiIiJHUlUgY2VsbCBtYXRjaGluZyBLZXJhcyBHUlUncyBjb25maWd1cmFibGUgY2FuZGlkYXRlIGFjdGl2YXRpb24uIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGlucHV0X2RpbTogaW50LCBoaWRkZW5fZGltOiBpbnQsIGFjdGl2YXRpb246IHN0ciA9ICJyZWx1IikgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmlucHV0X2RpbSA9IGludChpbnB1dF9kaW0pCiAgICAgICAgc2VsZi5oaWRkZW5fZGltID0gaW50KGhpZGRlbl9kaW0pCiAgICAgICAgc2VsZi5hY3RpdmF0aW9uX25hbWUgPSBhY3RpdmF0aW9uCiAgICAgICAgc2VsZi53ZWlnaHRfaWggPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkoMyAqIGhpZGRlbl9kaW0sIGlucHV0X2RpbSkpCiAgICAgICAgc2VsZi53ZWlnaHRfaGggPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkoMyAqIGhpZGRlbl9kaW0sIGhpZGRlbl9kaW0pKQogICAgICAgIHNlbGYuYmlhc19paCA9IG5uLlBhcmFtZXRlcih0b3JjaC5lbXB0eSgzICogaGlkZGVuX2RpbSkpCiAgICAgICAgc2VsZi5iaWFzX2hoID0gbm4uUGFyYW1ldGVyKHRvcmNoLmVtcHR5KDMgKiBoaWRkZW5fZGltKSkKICAgICAgICBzZWxmLnJlc2V0X3BhcmFtZXRlcnMoKQoKICAgIGRlZiByZXNldF9wYXJhbWV0ZXJzKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgbm4uaW5pdC54YXZpZXJfdW5pZm9ybV8oc2VsZi53ZWlnaHRfaWgpCiAgICAgICAgZm9yIGNodW5rIGluIHNlbGYud2VpZ2h0X2hoLmNodW5rKDMsIGRpbT0wKToKICAgICAgICAgICAgbm4uaW5pdC5vcnRob2dvbmFsXyhjaHVuaykKICAgICAgICBmYW5faW4gPSBzZWxmLmlucHV0X2RpbSArIHNlbGYuaGlkZGVuX2RpbQogICAgICAgIGJvdW5kID0gMS4wIC8gbWF0aC5zcXJ0KGZhbl9pbikKICAgICAgICBubi5pbml0LnVuaWZvcm1fKHNlbGYuYmlhc19paCwgLWJvdW5kLCBib3VuZCkKICAgICAgICBubi5pbml0LnVuaWZvcm1fKHNlbGYuYmlhc19oaCwgLWJvdW5kLCBib3VuZCkKCiAgICBkZWYgX2NhbmRpZGF0ZV9hY3RpdmF0aW9uKHNlbGYsIHZhbHVlOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBpZiBzZWxmLmFjdGl2YXRpb25fbmFtZSA9PSAicmVsdSI6CiAgICAgICAgICAgIHJldHVybiB0b3JjaC5yZWx1KHZhbHVlKQogICAgICAgIGlmIHNlbGYuYWN0aXZhdGlvbl9uYW1lID09ICJ0YW5oIjoKICAgICAgICAgICAgcmV0dXJuIHRvcmNoLnRhbmgodmFsdWUpCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuc3VwcG9ydGVkIEdSVSBhY3RpdmF0aW9uOiB7c2VsZi5hY3RpdmF0aW9uX25hbWV9IikKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IsIGhfcHJldjogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgZ2kgPSB0b3JjaC5tYXRtdWwoeCwgc2VsZi53ZWlnaHRfaWgudCgpKSArIHNlbGYuYmlhc19paAogICAgICAgIGdoID0gdG9yY2gubWF0bXVsKGhfcHJldiwgc2VsZi53ZWlnaHRfaGgudCgpKSArIHNlbGYuYmlhc19oaAogICAgICAgIGlfeiwgaV9yLCBpX24gPSBnaS5jaHVuaygzLCBkaW09LTEpCiAgICAgICAgaF96LCBoX3IsIGhfbiA9IGdoLmNodW5rKDMsIGRpbT0tMSkKCiAgICAgICAgeiA9IHRvcmNoLnNpZ21vaWQoaV96ICsgaF96KQogICAgICAgIHIgPSB0b3JjaC5zaWdtb2lkKGlfciArIGhfcikKICAgICAgICBuID0gc2VsZi5fY2FuZGlkYXRlX2FjdGl2YXRpb24oaV9uICsgciAqIGhfbikKICAgICAgICByZXR1cm4geiAqIGhfcHJldiArICgxLjAgLSB6KSAqIG4KCgpjbGFzcyBSZUxVR1JVTGF5ZXIobm4uTW9kdWxlKToKICAgICIiIlNpbmdsZS1kaXJlY3Rpb24gYmF0Y2gtZmlyc3QgR1JVIGxheWVyIHdpdGggUmVMVSBjYW5kaWRhdGUgYWN0aXZhdGlvbi4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5wdXRfZGltOiBpbnQsIGhpZGRlbl9kaW06IGludCwgcmV0dXJuX3NlcXVlbmNlczogYm9vbCkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmhpZGRlbl9kaW0gPSBpbnQoaGlkZGVuX2RpbSkKICAgICAgICBzZWxmLnJldHVybl9zZXF1ZW5jZXMgPSBib29sKHJldHVybl9zZXF1ZW5jZXMpCiAgICAgICAgc2VsZi5jZWxsID0gUmVMVUdSVUNlbGwoaW5wdXRfZGltPWlucHV0X2RpbSwgaGlkZGVuX2RpbT1oaWRkZW5fZGltLCBhY3RpdmF0aW9uPSJyZWx1IikKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBiYXRjaF9zaXplLCBzdGVwcywgXyA9IHguc2hhcGUKICAgICAgICBoID0geC5uZXdfemVyb3MoYmF0Y2hfc2l6ZSwgc2VsZi5oaWRkZW5fZGltKQogICAgICAgIG91dHB1dHMgPSBbXQogICAgICAgIGZvciBzdGVwIGluIHJhbmdlKHN0ZXBzKToKICAgICAgICAgICAgaCA9IHNlbGYuY2VsbCh4WzosIHN0ZXAsIDpdLCBoKQogICAgICAgICAgICBpZiBzZWxmLnJldHVybl9zZXF1ZW5jZXM6CiAgICAgICAgICAgICAgICBvdXRwdXRzLmFwcGVuZChoKQogICAgICAgIGlmIHNlbGYucmV0dXJuX3NlcXVlbmNlczoKICAgICAgICAgICAgcmV0dXJuIHRvcmNoLnN0YWNrKG91dHB1dHMsIGRpbT0xKQogICAgICAgIHJldHVybiBoCgoKZGVmIF9iYXRjaF9ub3JtX3RpbWUoYmF0Y2hfbm9ybTogbm4uQmF0Y2hOb3JtMWQsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgcmV0dXJuIGJhdGNoX25vcm0oeC50cmFuc3Bvc2UoMSwgMikpLnRyYW5zcG9zZSgxLCAyKQoKCmNsYXNzIEFkaUdSVU1vZGVsKG5uLk1vZHVsZSk6CiAgICAiIiJUd28tbGF5ZXIgUmVMVS1HUlUgbmV0d29yayBhZGFwdGVkIGZyb20gQWRpIGV0IGFsLidzIHBhcGVyLiIiIgoKICAgIHRhcmdldF9mcmFtZXMgPSBUQVJHRVRfRlJBTUVTCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5wdXRfZGltOiBpbnQgPSAxODAsCiAgICAgICAgbnVtX2NsYXNzZXM6IGludCA9IDEwLAogICAgICAgIGRyb3BvdXQ6IGZsb2F0ID0gREVGQVVMVF9EUk9QT1VULAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuaW5wdXRfZGltID0gaW50KGlucHV0X2RpbSkKICAgICAgICBzZWxmLm51bV9jbGFzc2VzID0gaW50KG51bV9jbGFzc2VzKQoKICAgICAgICBzZWxmLmdydTEgPSBSZUxVR1JVTGF5ZXIoc2VsZi5pbnB1dF9kaW0sIDY0LCByZXR1cm5fc2VxdWVuY2VzPVRydWUpCiAgICAgICAgc2VsZi5ibjEgPSBubi5CYXRjaE5vcm0xZCg2NCkKICAgICAgICBzZWxmLmRyb3BvdXQxID0gbm4uRHJvcG91dChmbG9hdChkcm9wb3V0KSkKCiAgICAgICAgc2VsZi5ncnUyID0gUmVMVUdSVUxheWVyKDY0LCAxMjgsIHJldHVybl9zZXF1ZW5jZXM9RmFsc2UpCiAgICAgICAgc2VsZi5ibjIgPSBubi5CYXRjaE5vcm0xZCgxMjgpCiAgICAgICAgc2VsZi5kcm9wb3V0MiA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dCkpCgogICAgICAgIHNlbGYuZGVuc2UgPSBubi5MaW5lYXIoMTI4LCA2NCkKICAgICAgICBzZWxmLmRyb3BvdXQzID0gbm4uRHJvcG91dChmbG9hdChkcm9wb3V0KSkKICAgICAgICBzZWxmLmNsYXNzaWZpZXIgPSBubi5MaW5lYXIoNjQsIHNlbGYubnVtX2NsYXNzZXMpCiAgICAgICAgc2VsZi5hY3RpdmF0aW9uID0gbm4uUmVMVSgpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgeCA9IHNlbGYuZ3J1MSh4KQogICAgICAgIHggPSBfYmF0Y2hfbm9ybV90aW1lKHNlbGYuYm4xLCB4KQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQxKHgpCgogICAgICAgIHggPSBzZWxmLmdydTIoeCkKICAgICAgICB4ID0gc2VsZi5ibjIoeCkKICAgICAgICB4ID0gc2VsZi5kcm9wb3V0Mih4KQoKICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UoeCkpCiAgICAgICAgeCA9IHNlbGYuZHJvcG91dDMoeCkKICAgICAgICByZXR1cm4gc2VsZi5jbGFzc2lmaWVyKHgpCgoKZGVmIGJ1aWxkX21vZGVsKGlucHV0X2RpbTogaW50LCBudW1fY2xhc3NlczogaW50KSAtPiBBZGlHUlVNb2RlbDoKICAgIHJldHVybiBBZGlHUlVNb2RlbChpbnB1dF9kaW09aW5wdXRfZGltLCBudW1fY2xhc3Nlcz1udW1fY2xhc3NlcykK',
    'gru_khukuh.py': 'IiIiS2h1a3VoLXN0eWxlIEdSVSBhcmNoaXRlY3R1cmUgZm9yIFNtYXJ0IFY4IEJJU0lORE8gZmVhdHVyZXMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KCgpUQVJHRVRfRlJBTUVTID0gMzAKREVGQVVMVF9MUiA9IDFlLTQKREVGQVVMVF9CQVRDSF9TSVpFID0gNjQKREVGQVVMVF9FUE9DSFMgPSAxMDAKREVGQVVMVF9EUk9QT1VUX0lOUFVUID0gMC4xMApERUZBVUxUX0RST1BPVVRfQkxPQ0sgPSAwLjIwCgoKY2xhc3MgS2h1a3VoR1JVTW9kZWwobm4uTW9kdWxlKToKICAgICIiIlRocmVlLXN0YWdlIEdSVSBuZXR3b3JrIGFkYXB0ZWQgZnJvbSBLaHVrdWggUHJpaGF0bWlraG8ncyBwYXBlci4KCiAgICBUaGUgcGFwZXIgdXNlcyAzMCBmcmFtZXMgYW5kIDE2MjkgTWVkaWFQaXBlIEhvbGlzdGljIHZhbHVlcyBwZXIgZnJhbWUuIFRoaXMKICAgIHZlcnNpb24gcHJlc2VydmVzIHRoZSB0ZW1wb3JhbCBsYXlvdXQgYW5kIGxheWVyIHNpemVzLCB3aGlsZSBhY2NlcHRpbmcgdGhlCiAgICBsb2NhbCBTbWFydCBWOCAxODAtRCBmZWF0dXJlIGNvbnRyYWN0IHVzZWQgYnkgdGhlIGN1cnJlbnQgZGF0YXNldC4KICAgICIiIgoKICAgIHRhcmdldF9mcmFtZXMgPSBUQVJHRVRfRlJBTUVTCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgaW5wdXRfZGltOiBpbnQgPSAxODAsCiAgICAgICAgbnVtX2NsYXNzZXM6IGludCA9IDEwLAogICAgICAgIGRyb3BvdXRfaW5wdXQ6IGZsb2F0ID0gREVGQVVMVF9EUk9QT1VUX0lOUFVULAogICAgICAgIGRyb3BvdXRfYmxvY2s6IGZsb2F0ID0gREVGQVVMVF9EUk9QT1VUX0JMT0NLLAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuaW5wdXRfZGltID0gaW50KGlucHV0X2RpbSkKICAgICAgICBzZWxmLm51bV9jbGFzc2VzID0gaW50KG51bV9jbGFzc2VzKQoKICAgICAgICBzZWxmLmlucHV0X2Ryb3BvdXQgPSBubi5Ecm9wb3V0KGZsb2F0KGRyb3BvdXRfaW5wdXQpKQoKICAgICAgICBzZWxmLmdydTEgPSBubi5HUlUoc2VsZi5pbnB1dF9kaW0sIDEyOCwgYmF0Y2hfZmlyc3Q9VHJ1ZSkKICAgICAgICBzZWxmLmRlbnNlMSA9IG5uLkxpbmVhcigxMjgsIDY0KQogICAgICAgIHNlbGYuZHJvcG91dDEgPSBubi5Ecm9wb3V0KGZsb2F0KGRyb3BvdXRfYmxvY2spKQoKICAgICAgICBzZWxmLmdydTIgPSBubi5HUlUoNjQsIDY0LCBiYXRjaF9maXJzdD1UcnVlKQogICAgICAgIHNlbGYuZGVuc2UyID0gbm4uTGluZWFyKDY0LCA2NCkKICAgICAgICBzZWxmLmRyb3BvdXQyID0gbm4uRHJvcG91dChmbG9hdChkcm9wb3V0X2Jsb2NrKSkKCiAgICAgICAgc2VsZi5ncnUzID0gbm4uR1JVKDY0LCAzMiwgYmF0Y2hfZmlyc3Q9VHJ1ZSkKICAgICAgICBzZWxmLmRlbnNlMyA9IG5uLkxpbmVhcigzMiwgNjQpCiAgICAgICAgc2VsZi5kcm9wb3V0MyA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dF9ibG9jaykpCgogICAgICAgIHNlbGYuYmF0Y2hfbm9ybSA9IG5uLkJhdGNoTm9ybTFkKDY0KQogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IG5uLkxpbmVhcig2NCwgc2VsZi5udW1fY2xhc3NlcykKICAgICAgICBzZWxmLmFjdGl2YXRpb24gPSBubi5SZUxVKCkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICB4ID0gc2VsZi5pbnB1dF9kcm9wb3V0KHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTEoeCkKICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UxKHgpKQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQxKHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTIoeCkKICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UyKHgpKQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQyKHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTMoeCkKICAgICAgICB4ID0geFs6LCAtMSwgOl0KCiAgICAgICAgeCA9IHNlbGYuYWN0aXZhdGlvbihzZWxmLmRlbnNlMyh4KSkKICAgICAgICB4ID0gc2VsZi5kcm9wb3V0Myh4KQogICAgICAgIHggPSBzZWxmLmJhdGNoX25vcm0oeCkKICAgICAgICByZXR1cm4gc2VsZi5jbGFzc2lmaWVyKHgpCgoKZGVmIGJ1aWxkX21vZGVsKGlucHV0X2RpbTogaW50LCBudW1fY2xhc3NlczogaW50KSAtPiBLaHVrdWhHUlVNb2RlbDoKICAgIHJldHVybiBLaHVrdWhHUlVNb2RlbChpbnB1dF9kaW09aW5wdXRfZGltLCBudW1fY2xhc3Nlcz1udW1fY2xhc3NlcykK',
    'gru_hybrid.py': 'IiIiSHlicmlkIEdSVSBhcmNoaXRlY3R1cmUgY29tYmluaW5nIEtodWt1aCBkZXB0aCB3aXRoIEFkaSByZWd1bGFyaXphdGlvbi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgoKClRBUkdFVF9GUkFNRVMgPSAzMApERUZBVUxUX0xSID0gMWUtNApERUZBVUxUX0JBVENIX1NJWkUgPSA2NApERUZBVUxUX0VQT0NIUyA9IDE1MApERUZBVUxUX0RST1BPVVRfSU5QVVQgPSAwLjEwCkRFRkFVTFRfRFJPUE9VVF9CTE9DSyA9IDAuMzAKREVGQVVMVF9QQVRJRU5DRSA9IDMwCgoKZGVmIF9iYXRjaF9ub3JtX3RpbWUoYmF0Y2hfbm9ybTogbm4uQmF0Y2hOb3JtMWQsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgcmV0dXJuIGJhdGNoX25vcm0oeC50cmFuc3Bvc2UoMSwgMikpLnRyYW5zcG9zZSgxLCAyKQoKCmNsYXNzIEh5YnJpZEdSVU1vZGVsKG5uLk1vZHVsZSk6CiAgICAiIiJGYXN0IDMwLWZyYW1lIGh5YnJpZCBmb3IgbGl2ZSB1c2UuCgogICAgVGhlIG1vZGVsIGtlZXBzIEtodWt1aCdzIDEyOC82NC8zMiByZWN1cnJlbnQgZGVwdGggYW5kIGluc2VydHMgQWRpLXN0eWxlCiAgICBCYXRjaE5vcm0vRHJvcG91dCBhZnRlciByZWN1cnJlbnQgYmxvY2tzIGZvciBiZXR0ZXIgc3RhYmlsaXR5IG9uIHRoZSBzbWFsbGVyCiAgICBsb2NhbCBkYXRhc2V0LgogICAgIiIiCgogICAgdGFyZ2V0X2ZyYW1lcyA9IFRBUkdFVF9GUkFNRVMKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBpbnB1dF9kaW06IGludCA9IDE4MCwKICAgICAgICBudW1fY2xhc3NlczogaW50ID0gMTAsCiAgICAgICAgZHJvcG91dF9pbnB1dDogZmxvYXQgPSBERUZBVUxUX0RST1BPVVRfSU5QVVQsCiAgICAgICAgZHJvcG91dF9ibG9jazogZmxvYXQgPSBERUZBVUxUX0RST1BPVVRfQkxPQ0ssCiAgICApIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5pbnB1dF9kaW0gPSBpbnQoaW5wdXRfZGltKQogICAgICAgIHNlbGYubnVtX2NsYXNzZXMgPSBpbnQobnVtX2NsYXNzZXMpCgogICAgICAgIHNlbGYuaW5wdXRfZHJvcG91dCA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dF9pbnB1dCkpCgogICAgICAgIHNlbGYuZ3J1MSA9IG5uLkdSVShzZWxmLmlucHV0X2RpbSwgMTI4LCBiYXRjaF9maXJzdD1UcnVlKQogICAgICAgIHNlbGYuYm4xID0gbm4uQmF0Y2hOb3JtMWQoMTI4KQogICAgICAgIHNlbGYuZGVuc2UxID0gbm4uTGluZWFyKDEyOCwgNjQpCiAgICAgICAgc2VsZi5kcm9wb3V0MSA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dF9ibG9jaykpCgogICAgICAgIHNlbGYuZ3J1MiA9IG5uLkdSVSg2NCwgNjQsIGJhdGNoX2ZpcnN0PVRydWUpCiAgICAgICAgc2VsZi5ibjIgPSBubi5CYXRjaE5vcm0xZCg2NCkKICAgICAgICBzZWxmLmRlbnNlMiA9IG5uLkxpbmVhcig2NCwgNjQpCiAgICAgICAgc2VsZi5kcm9wb3V0MiA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dF9ibG9jaykpCgogICAgICAgIHNlbGYuZ3J1MyA9IG5uLkdSVSg2NCwgMzIsIGJhdGNoX2ZpcnN0PVRydWUpCiAgICAgICAgc2VsZi5kZW5zZTMgPSBubi5MaW5lYXIoMzIsIDY0KQogICAgICAgIHNlbGYuYm4zID0gbm4uQmF0Y2hOb3JtMWQoNjQpCiAgICAgICAgc2VsZi5kcm9wb3V0MyA9IG5uLkRyb3BvdXQoZmxvYXQoZHJvcG91dF9ibG9jaykpCgogICAgICAgIHNlbGYuYWN0aXZhdGlvbiA9IG5uLlJlTFUoKQogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IG5uLkxpbmVhcig2NCwgc2VsZi5udW1fY2xhc3NlcykKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICB4ID0gc2VsZi5pbnB1dF9kcm9wb3V0KHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTEoeCkKICAgICAgICB4ID0gX2JhdGNoX25vcm1fdGltZShzZWxmLmJuMSwgeCkKICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UxKHgpKQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQxKHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTIoeCkKICAgICAgICB4ID0gX2JhdGNoX25vcm1fdGltZShzZWxmLmJuMiwgeCkKICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UyKHgpKQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQyKHgpCgogICAgICAgIHgsIF8gPSBzZWxmLmdydTMoeCkKICAgICAgICB4ID0geFs6LCAtMSwgOl0KICAgICAgICB4ID0gc2VsZi5hY3RpdmF0aW9uKHNlbGYuZGVuc2UzKHgpKQogICAgICAgIHggPSBzZWxmLmJuMyh4KQogICAgICAgIHggPSBzZWxmLmRyb3BvdXQzKHgpCiAgICAgICAgcmV0dXJuIHNlbGYuY2xhc3NpZmllcih4KQoKCmRlZiBidWlsZF9tb2RlbChpbnB1dF9kaW06IGludCwgbnVtX2NsYXNzZXM6IGludCkgLT4gSHlicmlkR1JVTW9kZWw6CiAgICByZXR1cm4gSHlicmlkR1JVTW9kZWwoaW5wdXRfZGltPWlucHV0X2RpbSwgbnVtX2NsYXNzZXM9bnVtX2NsYXNzZXMpCg==',
    'gru_manager.py': '...',
    'jetson_runtime.py': 'IiIiQ29sYWIgc3R1YiBvZiBqZXRzb25fcnVudGltZS4gVGhlIEdSVSB0cmFpbmluZyBwYXRoIG5ldmVyIGNhbGxzIHRoZXNlLgoKT25seSBwcmVzZW50IHNvIGBpbXBvcnQgamV0c29uX3J1bnRpbWUgYXMganJgIGluIGdydV9tYW5hZ2VyLnB5IHN1Y2NlZWRzIHVuY2hhbmdlZCwKYW5kIHRoZSBub24tdHJhaW5pbmcgaGVscGVycyAoYmVuY2htYXJrL3N0YXR1cykgZGVncmFkZSBncmFjZWZ1bGx5LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKCmRlZiBzZWxlY3RfbGl2ZV9kZXZpY2UodmFyaWFudD1Ob25lLCByZXF1ZXN0ZWQ9ImF1dG8iLCB0b3JjaF9tb2R1bGU9Tm9uZSk6CiAgICBuYW1lID0gImNwdSIKICAgIHRyeToKICAgICAgICBpZiB0b3JjaF9tb2R1bGUgaXMgbm90IE5vbmUgYW5kIHJlcXVlc3RlZCBpbiAoImF1dG8iLCAiY3VkYSIpIGFuZCB0b3JjaF9tb2R1bGUuY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgbmFtZSA9ICJjdWRhIgogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBuYW1lID0gImNwdSIKICAgIHJldHVybiBuYW1lLCAiY29sYWItc3R1YiIKCgpkZWYgZGlhZ25vc3RpY3ModG9yY2hfbW9kdWxlPU5vbmUpOgogICAgaW5mbyA9IHsicGxhdGZvcm0iOiAiY29sYWIiLCAid2FybmluZyI6IE5vbmV9CiAgICB0cnk6CiAgICAgICAgaWYgdG9yY2hfbW9kdWxlIGlzIG5vdCBOb25lOgogICAgICAgICAgICBpbmZvWyJjdWRhIl0gPSBib29sKHRvcmNoX21vZHVsZS5jdWRhLmlzX2F2YWlsYWJsZSgpKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBpbmZvWyJjdWRhIl0gPSBGYWxzZQogICAgcmV0dXJuIGluZm8KCgpkZWYgZGlhZ25vc3RpY3NfdGV4dChkaWFnPU5vbmUpOgogICAgcmV0dXJuIGYiamV0c29uX3J1bnRpbWUoY29sYWItc3R1Yik6IHtkaWFnIG9yIHt9fSIKCgpkZWYgaXNfamV0c29uX3BsYXRmb3JtKCk6CiAgICByZXR1cm4gRmFsc2UK',
}
for rel, b64 in EMBEDDED.items():
    path = os.path.join(SRC_ROOT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        f.write(base64.b64decode(b64))
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)
print("Source modul siap di", SRC_ROOT)
print("Files:", sorted(EMBEDDED.keys()))


In [ ]:
#@title 4. Konfigurasi training { display-mode: "form" }
SCHEMA = "smart180_face1584"  #@param ["smart180_face1584", "smart180", "khukuh1629", "adi1662"]
VARIANT = "adi"  #@param ["adi", "khukuh", "hybrid", "all"]
TRAIN_DATA = "with_augmentation"  #@param ["original", "with_augmentation", "both"]
DRIVE_DATASET_DIR = "/content/drive/MyDrive/dataset_parquets"  #@param {type:"string"}
DRIVE_MODEL_DIR = "/content/drive/MyDrive/bisindo_models"  #@param {type:"string"}
DEVICE = "auto"  #@param ["auto", "cuda", "cpu"]
COPY_TO_LOCAL = True  #@param {type:"boolean"}
OVERWRITE_EXISTING = True  #@param {type:"boolean"}
# Override hyperparameter (biarkan 0 / 0.0 = pakai default varian, persis repo)
EPOCHS = 0  #@param {type:"integer"}
BATCH_SIZE = 0  #@param {type:"integer"}
LR = 0.0  #@param {type:"number"}
PATIENCE = 0  #@param {type:"integer"}
LIMIT_PER_CLASS = 0  #@param {type:"integer"}
print("Config:", dict(schema=SCHEMA, variant=VARIANT, train_data=TRAIN_DATA, device=DEVICE,
                      copy_to_local=COPY_TO_LOCAL, overwrite=OVERWRITE_EXISTING))


In [ ]:
#@title 5. Siapkan dataset (copy Drive -> disk lokal biar I/O kenceng)
import os, shutil, glob, time

src_schema_dir = os.path.join(DRIVE_DATASET_DIR, SCHEMA)
assert os.path.isdir(src_schema_dir), f"Folder schema tidak ada di Drive: {src_schema_dir}"

if COPY_TO_LOCAL:
    dst_root = "/content/dataset_parquets"
    dst_schema_dir = os.path.join(dst_root, SCHEMA)
    os.makedirs(dst_schema_dir, exist_ok=True)
    files = sorted(glob.glob(os.path.join(src_schema_dir, "*.parquet")))
    assert files, f"Tidak ada *.parquet di {src_schema_dir}"
    t0 = time.time()
    for i, fp in enumerate(files, 1):
        dst = os.path.join(dst_schema_dir, os.path.basename(fp))
        if (not os.path.exists(dst)) or os.path.getsize(dst) != os.path.getsize(fp):
            shutil.copy2(fp, dst)
        if i % 10 == 0 or i == len(files):
            print(f"  copy {i}/{len(files)}")
    DATASET_DIR = dst_root
    print(f"Selesai copy {len(files)} parquet dalam {time.time()-t0:.1f}s -> {dst_schema_dir}")
else:
    DATASET_DIR = DRIVE_DATASET_DIR
    print("Pakai dataset langsung dari Drive:", DATASET_DIR)

n = len(glob.glob(os.path.join(DATASET_DIR, SCHEMA, "*.parquet")))
print("Jumlah parquet schema", SCHEMA, "=", n)


In [ ]:
#@title 6. Training (replika dispatch CLI `_cmd_train`)
import torch
import gru_manager as gm

print("Device:", "cuda" if torch.cuda.is_available() else "cpu",
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-only"))

MODEL_DIR_LOCAL = "/content/models"  # ROOT_DIR/models lokal, fresh tiap sesi
variants = gm.expand_variant_request(VARIANT, TRAIN_DATA)
print("Varian yang dilatih:", variants)

results = {}
for variant in variants:
    print("\n" + "=" * 70)
    print("TRAIN:", variant, "| schema:", SCHEMA)
    print("=" * 70)
    ok, msg = gm.train_variant(
        variant,
        dataset_dir=DATASET_DIR,
        model_dir=MODEL_DIR_LOCAL,
        schema=SCHEMA,
        epochs=(int(EPOCHS) or None),
        batch_size=(int(BATCH_SIZE) or None),
        lr=(float(LR) or None),
        patience=(int(PATIENCE) if int(PATIENCE) else None),
        device=DEVICE,
        limit_per_class=(int(LIMIT_PER_CLASS) or None),
        overwrite_existing=bool(OVERWRITE_EXISTING),
        train_data=gm.variant_train_data_mode(variant),
    )
    results[variant] = (ok, msg)
    print(msg)

print("\nRingkasan:")
for v, (ok, msg) in results.items():
    print(f"  [{'OK' if ok else 'GAGAL'}] {v}: {msg}")


In [ ]:
#@title 7. Simpan checkpoint ke Google Drive + tampilkan metadata
import os, shutil, glob, json

local_schema_models = os.path.join("/content/models", "gru", SCHEMA)
drive_schema_models = os.path.join(DRIVE_MODEL_DIR, "gru", SCHEMA)
os.makedirs(drive_schema_models, exist_ok=True)

copied = []
for fp in sorted(glob.glob(os.path.join(local_schema_models, "*"))):
    if os.path.isfile(fp):
        dst = os.path.join(drive_schema_models, os.path.basename(fp))
        shutil.copy2(fp, dst)
        copied.append(dst)

print("Tersimpan ke Drive:")
for c in copied:
    print("  ", c)

for fp in sorted(glob.glob(os.path.join(local_schema_models, "*_metadata.json"))):
    meta = json.load(open(fp))
    print("\n== " + os.path.basename(fp) + " ==")
    for k in ["variant", "schema", "feature_schema", "feature_dim", "target_frames",
              "num_classes", "train_samples", "val_samples", "train_augmented_samples",
              "best_epoch", "best_val_acc", "epochs_run", "epochs_requested", "batch_size", "lr"]:
        print(f"  {k}: {meta.get(k)}")


## ✅ Selesai

Checkpoint tersimpan di `DRIVE_MODEL_DIR/gru/<schema>/`:
`gru_<variant>.pth`, `gru_<variant>_labels.json`, `gru_<variant>_metadata.json`.

**Pakai di repo Jetson:** copy folder itu ke `models/gru/<schema>/` di repo.
Pastikan `feature_schema` di metadata cocok dengan schema yang dipakai saat live.

**Tips kecepatan:**
- `khukuh` / `hybrid` jauh lebih cepat di GPU (pakai cuDNN `nn.GRU`).
- `adi` pakai custom ReLU-GRU (loop per-timestep) — benar tapi lebih lambat.
- Naikkan `BATCH_SIZE` untuk schema besar (`smart180_face1584`, 1584-D) kalau VRAM cukup.
